# 14 OpenTelemetry LLM Observability

Set up OTLP tracing and metrics with `setup_grafana_otlp()` and the
`InstrumentedLlamaCppClient`.

**What you will learn:**
- Initialize OpenTelemetry tracer and meter providers
- Create an instrumented client that emits gen_ai.* spans and metrics
- Run traced inference requests
- Inspect telemetry attributes

**Requirements:** Kaggle T4 x2 with a running llama-server. Optional:
Grafana Cloud OTLP endpoint for remote export.

## 1) Install

In [ ]:
!pip -q install git+https://github.com/llamatelemetry/llamatelemetry.git@v0.1.0

## 2) Initialize telemetry

`setup_grafana_otlp()` returns a `(tracer, meter)` tuple. It configures
OTLP exporters if `GRAFANA_OTLP_ENDPOINT` / `OTLP_ENDPOINT` is set in
the environment.

In [ ]:
from llamatelemetry.telemetry import setup_grafana_otlp

tracer, meter = setup_grafana_otlp(
    service_name="llamatelemetry",
    service_version="0.1.0",
    llama_server_url="http://127.0.0.1:8080",
    enable_llama_metrics=True,
)
print(f"Tracer: {tracer}")
print(f"Meter:  {meter}")

## 3) Create an instrumented client

`InstrumentedLlamaCppClient` automatically creates spans and records
metrics for every inference call. It uses `chat_completions()` (plural)
which accepts a payload dict.

In [ ]:
from llamatelemetry.telemetry.client import InstrumentedLlamaCppClient

client = InstrumentedLlamaCppClient(base_url="http://127.0.0.1:8080")

## 4) Run a traced inference

In [ ]:
resp = client.chat_completions({
    "model": "local-gguf",
    "messages": [{"role": "user", "content": "What is OpenTelemetry?"}],
    "max_tokens": 64,
})
print(resp.choices[0].message.content)

## 5) Run a traced completion

In [ ]:
resp2 = client.completions({
    "prompt": "Explain OTLP in one sentence:",
    "max_tokens": 32,
})
print(resp2)

## 6) Traced embeddings

In [ ]:
emb_resp = client.embeddings({
    "input": "Telemetry test",
    "model": "local-gguf",
})
print(f"Embedding dimensions: {len(emb_resp.data[0].embedding) if hasattr(emb_resp, 'data') else 'N/A'}")

## 7) Notes

- All spans are tagged with `gen_ai.*` semantic convention attributes
  (45 attributes defined in the SDK).
- 5 metrics are recorded: `gen_ai.client.token.usage`,
  `gen_ai.client.operation.duration`, etc.
- If OTLP endpoint is configured, spans and metrics are exported
  automatically to Grafana Cloud, Jaeger, or any OTLP-compatible backend.